# colab_07 — Microglia subset: cl44 resolution, sub-clustering, and substate definition

Carries the microglia half of the colab_05 glia subset forward. colab_05 gave the **astrocytes** a full residual-batch adjudication and left two microglia items open: **cl44** (a weak-margin microglia cluster, never resolved) and the fact that microglia never got the parallel characterization. This notebook (1) resolves cl44, (2) sub-clusters the ca. 55,104 microglia to expose substate structure, (3) defines pre-committed **DAM vs homeostatic** substate labels that **eval #1** consumes, and (4) runs a lighter integrity scan (no gated 2nd-scVI unless a subcluster trips it). Runs on standard runtime + **high-RAM** (loads the ca. 6.5 GB glia object); no GPU needed — there is no model training here.

**Lifecycle:** this is the scaffold (no outputs). Phase-1 (Sonnet + user) then Phase-2 (cold-Opus) review before running; interp cells are written against observed values into the `_OUTPUT` copy after the run.

## 1 — Setup

### 1a — Mount Drive + clone/pull repo + install env

Same pattern as colab_04–06: mount Drive, clone-or-pull the repo, pin numpy first, install the integration env. Adds an explicit `leidenalg` install (the §5 Leiden sub-clustering) in case it is not pinned in `requirements_integration.txt`. No GPU is requested — sub-clustering, scoring and DE are CPU work; the only heavy resource is RAM for the ca. 6.5 GB glia object.

In [ ]:
import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/ad-glia-fm-prep"
os.makedirs(DRIVE_ROOT, exist_ok=True)

REPO_URL  = "https://github.com/pavlemic/ad-glia-fm-prep.git"
REPO_PATH = "/content/ad-glia-fm-prep"

if not os.path.exists(REPO_PATH):
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)
else:
    subprocess.run(["git", "-C", REPO_PATH, "pull"], check=True)

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

assert sys.version_info[:2] == (3, 12), f"Expected Python 3.12, got {sys.version_info[:2]}"

# Pin numpy first so pip picks numpy-1.x-compatible wheels (same rationale as colab_01-06).
!pip install numpy==1.26.4
!pip install -r {REPO_PATH}/requirements_integration.txt
# Leiden sub-clustering (§5) needs leidenalg + python-igraph; ensure present even if unpinned.
!pip install leidenalg igraph

## 2 — Environment capture

### 2a — pip freeze + env JSON

Identical reproducibility snapshot to the earlier notebooks: full `pip freeze` plus a structured JSON of Python / OS / library versions and the repo commit, written under `outputs/software_versions/`. Methods sections need the exact versions present at run time, not the pin file.

In [ ]:
import json, platform, subprocess, sys
from datetime import date

NOTEBOOK_ID = "colab_07"
TODAY = date.today().isoformat()
VERSIONS_DIR = os.path.join(REPO_PATH, "outputs", "software_versions")
os.makedirs(VERSIONS_DIR, exist_ok=True)

FREEZE_PATH = os.path.join(VERSIONS_DIR, f"{NOTEBOOK_ID}_{TODAY}_pip_freeze.txt")
!pip freeze > {FREEZE_PATH}

def _run(cmd):
    try:
        return subprocess.run(cmd, capture_output=True, text=True, check=True).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        return None

def _ver(mod):
    try:
        return __import__(mod).__version__
    except Exception:
        return None

env_snapshot = {
    "notebook_id":   NOTEBOOK_ID,
    "date":          TODAY,
    "python_version": sys.version,
    "platform":      platform.platform(),
    "os_release":    platform.release(),
    "gpu":           _run(["nvidia-smi", "-L"]),
    "git_commit":    _run(["git", "-C", REPO_PATH, "rev-parse", "HEAD"]),
    "scanpy_version":   _ver("scanpy"),
    "anndata_version":  _ver("anndata"),
    "leidenalg_version":_ver("leidenalg"),
}
ENV_JSON_PATH = os.path.join(VERSIONS_DIR, f"{NOTEBOOK_ID}_{TODAY}_env.json")
with open(ENV_JSON_PATH, "w") as f:
    json.dump(env_snapshot, f, indent=2)
print(json.dumps(env_snapshot, indent=2))

## 3 — Load the glia subset + carve out microglia

### 3a — Load `glia_subset_full.h5ad`, subset microglia, sanity-check latent + raw counts

Load the colab_05 glia hand-off object (astrocytes + microglia, 149,375 cells; carries raw counts in `.X`, the scVI latent `X_scVI`, the original `leiden_scvi` clusters, and `cell_type`). Fail loud unless those are present and `.X` is integer counts. Then carve out the microglia (`cell_type == "microglia"`, ca. 55,104 cells) — perivascular macrophages (old cl0) were split into their own `cell_type` in colab_05 and so are already excluded. Print the microglia cluster × study table, donor count, and APOE breakdown so the working set is characterized before anything is done to it.

In [ ]:
import gc
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import scipy.sparse as sp
import matplotlib.pyplot as plt

try:
    import psutil
    def _ram(tag):
        m = psutil.virtual_memory()
        print(f"[RAM] {tag:24s}: {m.used/1e9:5.1f} / {m.total/1e9:.1f} GB ({m.percent:.0f}%)")
except ImportError:
    def _ram(tag): pass

sc.settings.verbosity = 1
FIG_DIR = os.path.join(REPO_PATH, "figures", "colab_07")
os.makedirs(FIG_DIR, exist_ok=True)

GLIA_PATH = os.path.join(DRIVE_ROOT, "glia_subset", "glia_subset_full.h5ad")
if not os.path.exists(GLIA_PATH):
    raise FileNotFoundError(f"missing colab_05 glia subset {GLIA_PATH}")
glia = sc.read_h5ad(GLIA_PATH)
print("loaded glia subset:", glia.shape)

assert "X_scVI" in glia.obsm, "X_scVI missing — colab_05 glia subset expected"
assert "leiden_scvi" in glia.obs.columns, "leiden_scvi missing — needed for cl44 resolution"
assert "cell_type" in glia.obs.columns, "cell_type missing — needed to carve microglia"

# raw-counts guard (sorted deposits hide non-integer tails in a head slice -> random sample)
_idx = np.random.default_rng(0).choice(glia.n_obs, size=min(2000, glia.n_obs), replace=False)
Xs = glia.X[_idx]
data = Xs.data if sp.issparse(Xs) else np.asarray(Xs).ravel()
frac_int = float(np.mean(np.mod(data, 1) == 0)) if data.size else 1.0
assert frac_int >= 0.99, f".X is not raw counts (int frac {frac_int:.3f})"

# carve out microglia (cl0 = perivascular_mac was already split off in colab_05 -> not "microglia")
micro = glia[glia.obs["cell_type"] == "microglia"].copy()
micro.obs["leiden_scvi"] = micro.obs["leiden_scvi"].astype(str)
del glia; gc.collect()
print("microglia subset:", micro.shape)
print("\nmicroglia leiden_scvi x study (counts):")
print(pd.crosstab(micro.obs["leiden_scvi"], micro.obs["study_id"]))
print("\ndonors:", micro.obs["donor_id"].nunique(),
      "| apoe_carrier:", micro.obs["apoe_carrier"].value_counts(dropna=False).to_dict())
_ram("microglia subset")

## 4 — Resolve cl44 (the weak-margin microglia cluster)

### 4a — cl44 composition, marker re-score, and DE vs homeostatic / non-microglia

cl44 was the weakest-margin microglia cluster in colab_05's annotation (its top-two lineage scores were close), so before sub-clustering we settle the coarse question: **does cl44 belong in the microglia working set at all?** Three pieces of evidence:

- **Marker re-score** — recompute the canonical lineage signatures (microglia, PVM, oligo, astro, neuron) on the microglia subset and read cl44's per-lineage means and argmax margin. A confident microglia call has microglia >> everything else.
- **Differential expression** — cl44 vs cl1 (the homeostatic-microglia anchor) shows *how* cl44 differs from textbook microglia (an activated substate? a contaminant?); cl44 vs the rest of the microglia gives its standalone signature. (Wilcoxon on the microglia subset only — within the project's compute budget; never on the full object.)
- **Marker dotplot** — microglia (CSF1R/P2RY12), PVM (MRC1/CD163/LYVE1), and doublet/ambient (oligo MBP/PLP1, astro AQP4/GFAP, neuron SNAP25/RBFOX3) markers side by side, so a non-microglial identity shows up as a clear off-lineage band.

The verdict (keep as microglia / reassign / drop as doublet-ambient) is written in the `_OUTPUT` interp; this cell only assembles the evidence.

In [ ]:
# log-normalized layer for scoring (.X stays raw counts) — rebuilt on the micro subset
micro.layers["lognorm"] = micro.X.copy()
sc.pp.normalize_total(micro, target_sum=1e4, layer="lognorm")
micro.uns.pop("log1p", None)
sc.pp.log1p(micro, layer="lognorm")
micro.uns.pop("log1p", None)

# canonical lineage markers (colab_05/06 MARKERS, verbatim) for the coarse identity check
MARKERS = {
    "microglia":        ["CSF1R", "P2RY12", "TMEM119", "AIF1", "TREM2", "CX3CR1"],
    "perivascular_mac": ["MRC1", "CD163", "LYVE1", "F13A1"],
    "oligodendrocyte":  ["MBP", "MOBP", "PLP1", "MOG"],
    "astrocyte":        ["AQP4", "SLC1A2", "GFAP", "GJA1", "SLC1A3"],
    "neuron_exc":       ["RBFOX3", "SNAP25", "SLC17A7", "SATB2"],
}
present = {ct: [g for g in gs if g in micro.var_names] for ct, gs in MARKERS.items()}
for ct, gs in present.items():
    miss = set(MARKERS[ct]) - set(gs)
    if miss:
        print(f"[markers] {ct}: absent from gene set -> {sorted(miss)}")

X_raw = micro.X
score_cols = []
micro.X = micro.layers["lognorm"]
try:
    for ct, gs in present.items():
        if gs:
            sc.tl.score_genes(micro, gene_list=gs, score_name=f"score_{ct}", use_raw=False)
            score_cols.append(f"score_{ct}")
finally:
    micro.X = X_raw

# per micro-cluster mean lineage scores + argmax margin (which clusters are weak microglia?)
S = micro.obs.groupby("leiden_scvi", observed=True)[score_cols].mean()
S.columns = [c.replace("score_", "") for c in S.columns]
srt = np.sort(S.values, axis=1)
margin = pd.Series(srt[:, -1] - srt[:, -2], index=S.index).round(3)
print("per micro-cluster mean lineage scores:")
print(S.round(2))
print("\nargmax + margin (low margin = weak microglia identity):")
print(pd.DataFrame({"argmax": S.idxmax(axis=1), "margin": margin}).sort_values("margin"))

# cl44 focused inspection
CL44 = "44"
assert CL44 in S.index, "cl44 absent from microglia subset — revisit colab_05 annotation"
m44 = micro.obs["leiden_scvi"] == CL44
print(f"\ncl44: {int(m44.sum())} cells")
print("  study      :", micro.obs.loc[m44, "study_id"].value_counts().to_dict())
print("  top donors :", micro.obs.loc[m44, "donor_id"].value_counts().head(5).to_dict())
print("  lineage scores:", S.loc[CL44].round(3).to_dict(), "| margin", margin[CL44])

# DE: cl44 vs cl1 (homeostatic anchor) and vs the rest of microglia (micro-subset Wilcoxon)
micro.X = micro.layers["lognorm"]
try:
    sc.tl.rank_genes_groups(micro, "leiden_scvi", groups=[CL44], reference="1",
                            method="wilcoxon", key_added="cl44_vs_homeo")
    sc.tl.rank_genes_groups(micro, "leiden_scvi", groups=[CL44], reference="rest",
                            method="wilcoxon", key_added="cl44_vs_rest")
finally:
    micro.X = X_raw
print("\ncl44 vs cl1 (homeostatic) — top 15 up:")
print(pd.DataFrame(micro.uns["cl44_vs_homeo"]["names"])[CL44].head(15).tolist())
print("cl44 vs rest-of-microglia — top 15 up:")
print(pd.DataFrame(micro.uns["cl44_vs_rest"]["names"])[CL44].head(15).tolist())

# marker dotplot: microglia vs PVM vs doublet/ambient bands across all micro clusters
DOT_GENES = ["CSF1R","P2RY12","TMEM119","CX3CR1","TREM2","AIF1",
             "MRC1","CD163","LYVE1","MBP","PLP1","AQP4","GFAP","SNAP25","RBFOX3"]
dot_present = [g for g in DOT_GENES if g in micro.var_names]
micro.X = micro.layers["lognorm"]
try:
    sc.pl.dotplot(micro, dot_present, groupby="leiden_scvi", standard_scale="var", show=False)
    plt.savefig(os.path.join(FIG_DIR, "4a_microglia_cluster_marker_dotplot.png"),
                dpi=150, bbox_inches="tight")
    plt.close()
finally:
    micro.X = X_raw

## 5 — Microglia sub-clustering

### 5a — Neighbors + Leiden on `X_scVI` within microglia + UMAP + per-subcluster DE

The global 46-cluster solution barely split the microglia (they sat mostly in one cluster), so substate structure is invisible at that resolution. Re-running the neighbor graph and Leiden **on the microglia only** — on the same scVI latent used throughout (consistent with how colab_05 worked the astrocytes) — lets finer substates separate. A within-microglia re-embedding (fresh HVGs / PCA) is the fallback if substates do not resolve on the integration latent.

**Why the resolution matters here:** §6a assigns one substate label *per subcluster* (the mean DAM/homeostatic score over each Leiden group), so the partition is load-bearing — too coarse dilutes a real DAM pocket into "intermediate" by averaging it with homeostatic neighbors; too fine fragments one substate into noisy groups with unstable means. We keep cluster-level assignment (for its denoising) at a primary `resolution=1.0`, and report an ARI-stability check at res 0.5 / 1.5 so the post-run interp can judge how sensitive the partition is. The resolution-independent alternative — thresholding each cell's continuous score directly — was declined here because it forfeits that denoising and is noisier at single-cell level.

The UMAPs are colored by subcluster, study (to spot any subcluster that is one study only), the original `leiden_scvi` (to see where cl44's cells land), and APOE. Per-subcluster Wilcoxon DE gives each subcluster its top markers for the substate read in §6.

In [ ]:
# sub-cluster microglia on the scVI latent (consistent with colab_05's use of X_scVI for astro)
sc.pp.neighbors(micro, use_rep="X_scVI", n_neighbors=15, random_state=0)
sc.tl.leiden(micro, resolution=1.0, key_added="leiden_micro", random_state=0)
sc.tl.umap(micro, random_state=0)
print("micro subclusters:", micro.obs["leiden_micro"].nunique())
print(micro.obs["leiden_micro"].value_counts().sort_index())

# resolution-stability sanity-check: §6a assigns ONE substate label per subcluster (mean score),
# so the Leiden resolution is load-bearing. Report subcluster counts + ARI vs the primary res=1.0
# at neighbouring resolutions so the _OUTPUT interp can judge how stable the partition is
# (res=1.0 stays primary; per-cell score thresholding is the resolution-independent alternative
# we declined, to keep the cluster-level denoising).
from sklearn.metrics import adjusted_rand_score
print("\nresolution stability (vs res=1.0):")
for r in (0.5, 1.5):
    sc.tl.leiden(micro, resolution=r, key_added=f"_leiden_r{r}", random_state=0)
    ari = adjusted_rand_score(micro.obs["leiden_micro"], micro.obs[f"_leiden_r{r}"])
    print(f"  res={r}: {micro.obs[f'_leiden_r{r}'].nunique()} subclusters, ARI vs res=1.0 = {ari:.3f}")
    del micro.obs[f"_leiden_r{r}"]

print("\nleiden_micro x study (row-normalized):")
print(pd.crosstab(micro.obs["leiden_micro"], micro.obs["study_id"], normalize="index").round(3))
print("\nleiden_micro x original leiden_scvi (where cl44 lands):")
print(pd.crosstab(micro.obs["leiden_micro"], micro.obs["leiden_scvi"]))

# per-subcluster marker DE (micro-subset Wilcoxon — within budget per the project rule)
X_raw = micro.X
micro.X = micro.layers["lognorm"]
try:
    sc.tl.rank_genes_groups(micro, "leiden_micro", method="wilcoxon", key_added="micro_de")
finally:
    micro.X = X_raw
print("\ntop-8 markers per micro subcluster:")
print(pd.DataFrame(micro.uns["micro_de"]["names"]).head(8))

# UMAPs
for color in ["leiden_micro", "study_id", "leiden_scvi", "apoe_carrier"]:
    loc = "on data" if color in ("leiden_micro", "leiden_scvi") else "right margin"
    sc.pl.umap(micro, color=color, show=False, legend_loc=loc)
    plt.savefig(os.path.join(FIG_DIR, f"5a_umap_{color}.png"), dpi=150, bbox_inches="tight")
    plt.close()
_ram("after subclustering")

## 6 — Substate definition + integrity scan

### 6a — Score DAM vs homeostatic signatures → provisional substate labels (eval #1 source)

This closes the eval #1 OPEN item for microglia. The probe in eval #1 asks whether continued-pretraining sharpens **substate** structure, which needs substate labels that do **not** exist in the raw metadata. We define them from published marker signatures — **DAM** (disease-associated microglia: TREM2/TYROBP/APOE/CST7/LPL/ITGAX/CD9/SPP1/GPNMB/CTSB/CTSD) vs **homeostatic** (surveillant resting: P2RY12/CX3CR1/TMEM119/SALL1/P2RY13/SELPLG). These are scored on **raw-expression-derived** values, completely independent of any FM embedding — so labelling cells this way and later probing FM embeddings against the labels is **not** circular. The DAM↔homeostatic axis is a continuum, so a subcluster whose two scores are within a small margin is called `intermediate` rather than forced to a pole. The signature lists, the margin, and the per-subcluster assignment are written to `outputs/microglia_substate_signatures.json` as the committed definition (the contract requires substate labels be fixed and committed before the probe is built).

**Three caveats carried into the interp (decided at Phase-1, recorded so they are not silently load-bearing):**

- **DAM is the Keren-Shaul 2017 *mouse* AD-model core.** Many of its genes (TREM2, APOE, SPP1, GPNMB, CD9, CST7) recur in human-AD microglia activation reports, but mouse DAM staging does not transfer 1:1 to human microglia. It is kept for recognizability against the FM/AD literature, but treated as a limitation — if the axis fails to separate, "no DAM signal in this data" and "wrong gene list for human microglia" are not distinguishable. The post-run interp states this explicitly.
- **APOE appears in the DAM list (expression), and eval #2 probes APOE *genotype*.** These are not independent (genotype is an eQTL for APOE transcript level), so a joint eval #1 + eval #2 "win" must never be claimed as two separate confirmations. APOE is kept (canonical marker) and the non-independence is documented rather than dropped.
- **`INT_MARGIN = 0.05` is an unvalidated placeholder in `score_genes` units** (not the contract's silhouette/accuracy 0.05 — same number, different scale). It is calibrated once against the observed score-gap distribution post-run, the same "refine at pilot" pattern as the evaluation contract.

In [ ]:
# Pre-committed microglia substate signatures (published references), scored on raw-expression-
# derived lognorm — INDEPENDENT of any FM embedding, so eval#1 (probe FM embeddings vs these
# labels) is not circular. DAM = Keren-Shaul 2017 mouse-AD core. Many of these (TREM2, APOE,
# SPP1, GPNMB, CD9, CST7) recur in human-AD microglia activation reports, but the mouse DAM
# staging does NOT map 1:1 onto human microglia — treated as a recognizable-but-limited
# reference signature, not a validated human label (limitation recorded in the §6a interp).
# NOTE: APOE sits in the DAM list (expression level); eval#2 separately probes APOE *genotype*.
# These are NOT independent (genotype is an eQTL for APOE transcript), so a joint eval#1+eval#2
# "win" must never be read as two separate confirmations. Kept (canonical DAM marker) and
# documented rather than dropped to manufacture independence.
SUBSTATE_SIG = {
    "DAM":         ["TREM2","TYROBP","APOE","CST7","LPL","ITGAX","CD9","SPP1","GPNMB","CTSB","CTSD"],
    "homeostatic": ["P2RY12","CX3CR1","TMEM119","SALL1","P2RY13","SELPLG"],
}
sig_present = {k: [g for g in gs if g in micro.var_names] for k, gs in SUBSTATE_SIG.items()}
for k, gs in sig_present.items():
    miss = set(SUBSTATE_SIG[k]) - set(gs)
    if miss:
        print(f"[substate] {k}: absent from gene set -> {sorted(miss)}")

X_raw = micro.X
micro.X = micro.layers["lognorm"]
try:
    for k, gs in sig_present.items():
        sc.tl.score_genes(micro, gene_list=gs, score_name=f"sub_{k}", use_raw=False)
finally:
    micro.X = X_raw

sub_cols = [f"sub_{k}" for k in SUBSTATE_SIG]
SUB = micro.obs.groupby("leiden_micro", observed=True)[sub_cols].mean()
SUB.columns = [c.replace("sub_", "") for c in SUB.columns]
print("per-subcluster mean substate scores:")
print(SUB.round(3))

# provisional substate per subcluster: argmax of DAM vs homeostatic, with an 'intermediate' band.
# INT_MARGIN is in sc.tl.score_genes units (NOT the silhouette/accuracy 0.05 in EVALUATION_CONTRACT
# — same number, different scale). Unvalidated placeholder: calibrate ONCE against the observed
# score-gap distribution post-run (recorded in the §6a interp), same "refine at pilot" pattern as
# the contract — do NOT treat 0.05 here as pre-validated.
INT_MARGIN = 0.05
sub_assign = {}
for cl, row in SUB.iterrows():
    top = row.idxmax(); second = row.drop(top).idxmax()
    sub_assign[cl] = "intermediate" if (row[top] - row[second]) < INT_MARGIN else top
micro.obs["substate"] = micro.obs["leiden_micro"].map(sub_assign).astype(str)
print("\nprovisional substate per subcluster:", sub_assign)
print("substate cell counts:", micro.obs["substate"].value_counts().to_dict())

# committed signature definition (eval#1 OPEN item -> closed for microglia)
SIG_PATH = os.path.join(REPO_PATH, "outputs", "microglia_substate_signatures.json")
with open(SIG_PATH, "w") as f:
    json.dump({"date": TODAY, "lineage": "microglia", "signatures": SUBSTATE_SIG,
               "present": sig_present, "intermediate_margin": INT_MARGIN,
               "per_subcluster_assignment": sub_assign,
               "note": "expression-signature substate labels for the eval#1 probe; "
                       "independent of FM embeddings (not circular)"}, f, indent=2)
print("saved substate definition ->", SIG_PATH)

# substate score UMAP + signature-gene dotplot
micro.X = micro.layers["lognorm"]
try:
    sc.pl.umap(micro, color=["sub_DAM", "sub_homeostatic", "substate"], show=False)
    plt.savefig(os.path.join(FIG_DIR, "6a_substate_scores_umap.png"), dpi=150, bbox_inches="tight")
    plt.close()
    sig_genes = [g for gs in sig_present.values() for g in gs]
    sc.pl.dotplot(micro, sig_genes, groupby="leiden_micro", standard_scale="var", show=False)
    plt.savefig(os.path.join(FIG_DIR, "6a_substate_signature_dotplot.png"), dpi=150, bbox_inches="tight")
    plt.close()
finally:
    micro.X = X_raw

### 6b — Per-subcluster integrity scan (donor / study / ambient)

The lighter counterpart to colab_05's astrocyte battery. The astrocyte adjudication ran a full six-lever battery because a concrete residual-batch flag (a Li-pure ambient cluster) had been raised; no such flag was raised for microglia, so here we **report** the same per-subcluster diagnostics — dominant-donor fraction, dominant-study fraction, median counts / mito, and an ambient/doublet proxy (oligo / astro / neuron marker leakage) — without committing to the heavy gated 2nd-scVI. A subcluster that is donor-dominated (≥ 50%) or study-dominated (≥ 60%) is flagged for the interp; the gated 2nd-scVI is run **only** if a flagged subcluster also turns out to carry the APOE signal eval #2 depends on. This keeps the integrity claim defensible without process-for-its-own-sake.

In [ ]:
# lighter integrity scan: per-subcluster donor/study dominance + ambient/quality proxies
X_raw = micro.X
if "pct_counts_mt" not in micro.obs.columns or "total_counts" not in micro.obs.columns:
    micro.var["mt"] = micro.var_names.str.startswith("MT-")
    sc.pp.calculate_qc_metrics(micro, qc_vars=["mt"], inplace=True, percent_top=None, log1p=False)

AMBIENT = [g for g in ["MBP","PLP1","MOBP","AQP4","GFAP","SNAP25","RBFOX3"] if g in micro.var_names]
micro.X = micro.layers["lognorm"]
try:
    sc.tl.score_genes(micro, gene_list=AMBIENT, score_name="score_ambient", use_raw=False)
finally:
    micro.X = X_raw

rows = []
for cl, idx in micro.obs.groupby("leiden_micro", observed=True).groups.items():
    o = micro.obs.loc[idx]
    rows.append({
        "subcluster": cl, "n": len(o),
        "top_donor_frac": round(float(o["donor_id"].value_counts(normalize=True).iloc[0]), 3),
        "top_study_frac": round(float(o["study_id"].value_counts(normalize=True).iloc[0]), 3),
        "median_counts": int(o["total_counts"].median()),
        "median_pct_mt": round(float(o["pct_counts_mt"].median()), 2),
        "mean_ambient": round(float(o["score_ambient"].mean()), 3),
    })
scan = pd.DataFrame(rows).set_index("subcluster")
DONOR_FLAG, STUDY_FLAG = 0.50, 0.60
scan["flag"] = np.where(scan["top_donor_frac"] >= DONOR_FLAG, "donor",
                np.where(scan["top_study_frac"] >= STUDY_FLAG, "study", ""))
print("per-subcluster integrity scan:")
with pd.option_context("display.width", 160):
    print(scan)
flagged = scan[scan["flag"] != ""]
if len(flagged):
    print(f"\n[FLAG] {len(flagged)} subcluster(s) donor/study-dominated — report in interp; "
          f"gated 2nd-scVI ONLY if a flagged cluster also carries the eval#2 APOE signal:")
    print(flagged)
else:
    print("\nno subcluster donor/study-dominated above thresholds — microglia substrate clean")

## 7 — Save + handoff

### 7a — Save microglia subset with substate labels; audit trace; commit instructions

Save the microglia subset (raw counts in `.X` for the FM substrate, plus `leiden_micro`, `substate`, and the carried-over `apoe_carrier` / donor / study metadata) to Drive; the working `lognorm` layer is dropped before saving (rebuildable, keeps the file small). Append a `microglia_subset` trace to the cumulative `audit_report.json` and print the WSL commit commands. Committable = env freeze + audit JSON + the substate-signature JSON; the h5ad and figures are Drive- / gitignore-only.

In [ ]:
import shlex
# NOTE: if the §4 cl44 verdict reassigns/drops cl44, apply that membership change here BEFORE
# saving (it is left as keep-by-default in the scaffold; the _OUTPUT interp records the verdict).

MICRO_DIR = os.path.join(DRIVE_ROOT, "micro_subset")
os.makedirs(MICRO_DIR, exist_ok=True)
if "lognorm" in micro.layers:
    del micro.layers["lognorm"]                       # rebuildable; .X stays raw counts
if sp.issparse(micro.X) and micro.X.getformat() != "csr":
    micro.X = sp.csr_matrix(micro.X)
MICRO_PATH = os.path.join(MICRO_DIR, "micro_subset.h5ad")
micro.write_h5ad(MICRO_PATH)
print("saved microglia subset ->", MICRO_PATH, f"({os.path.getsize(MICRO_PATH)/1e9:.2f} GB)")

AUDIT_REPORT_PATH = os.path.join(REPO_PATH, "outputs", "audit_report.json")
with open(AUDIT_REPORT_PATH) as f:
    report = json.load(f)
report["microglia_subset"] = {
    "status": "computed",
    "date": TODAY,
    "n_microglia": int(micro.n_obs),
    "n_subclusters": int(micro.obs["leiden_micro"].nunique()),
    "subcluster_on": "X_scVI",
    "substate_counts": micro.obs["substate"].value_counts().to_dict(),
    "substate_signatures_file": os.path.relpath(SIG_PATH, REPO_PATH),
    "cl44_verdict": "<set from 4a interp before commit>",
}
with open(AUDIT_REPORT_PATH, "w") as f:
    json.dump(report, f, indent=2)
print("audit trace appended ->", AUDIT_REPORT_PATH)

rel = [os.path.relpath(p, REPO_PATH) for p in (FREEZE_PATH, ENV_JSON_PATH, AUDIT_REPORT_PATH, SIG_PATH)]
print("\n=== Commit + push (from WSL — Colab has no git creds) ===")
print("  cd /content/ad-glia-fm-prep && git add " + " ".join(shlex.quote(r) for r in rel))
print("  cd /content/ad-glia-fm-prep && git commit -m "
      "'colab_07: microglia subset — cl44 resolution, subclustering, substate labels'")
print("  cd /content/ad-glia-fm-prep && git push")

### Carried forward to the FM notebooks

- **Microglia subset** (`micro_subset.h5ad`, raw genes) — the microglia CPT substrate and the eval #1 / eval #2 microglia working set.
- **Substate labels** (`substate`: DAM / homeostatic / intermediate) + their committed signature definition — the eval #1 label source for microglia.
- **cl44 disposition** — recorded in the audit trace and applied to the saved subset.
- **Still open:** the **astrocyte** substate labels (DAA / resting) are not yet defined — colab_05 gave astrocytes their adjudication but not substate labels. eval #1 needs both lineages' substate labels committed before the probe, so an astrocyte-substate step is the natural next notebook.